In [1]:
import pandas as pd
from datetime import datetime, timedelta
import os
import numpy as np
import zipfile

In [2]:
path = r"C:\Users\Akhil\Downloads\project\TradeX_raw\STcon"


os.chdir(path)
print("Current Working Directory:", os.getcwd())


xlsx_files = [f for f in os.listdir(path) if f.endswith('.xlsx')]

if len(xlsx_files) != 2:
    raise ValueError("There should be exactly 2 .xlsx files in the directory.")

Current Working Directory: C:\Users\Akhil\Downloads\project\TradeX_raw\STcon


In [3]:


# Read the files
st_1 = pd.read_excel(xlsx_files[0])
st_2 = pd.read_excel(xlsx_files[1])

# Concatenate
concat_df = pd.concat([st_1, st_2], ignore_index=True)

print("Length of st_1:", len(st_1))
print("Length of st_2:", len(st_2))


print("Length of concatenated DataFrame:", len(concat_df))

# Filter out unwanted rows
stringee = concat_df[concat_df['End of Call code'] != 'CAN_NOT_MAKE_CALL']

# Generate dynamic filename S_DD_MM.csv
yesterday = datetime.today()-timedelta(days=1)
file_name = f"S_{yesterday.day:02d}_{yesterday.month:02d}.csv"

# Build full output path
output_path = os.path.join(
    r"C:\Users\Akhil\Downloads\project\TradeX_raw",
    file_name
)

# Save CSV
stringee.to_csv(output_path, index=False)
print(f"CSV saved to: {output_path}")


c:\Users\Akhil\Office\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Akhil\Office\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Length of st_1: 0
Length of st_2: 2502
Length of concatenated DataFrame: 2502
CSV saved to: C:\Users\Akhil\Downloads\project\TradeX_raw\S_04_04.csv


C:\Users\Akhil\AppData\Local\Temp\ipykernel_1400\3173678099.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_df = pd.concat([st_1, st_2], ignore_index=True)


In [4]:
path = r"C:\Users\Akhil\Downloads\project\TradeX_raw\TTcon"

# Change the working directory
os.chdir(path)

# Confirm the new working directory
print("Current Working Directory:", os.getcwd())

Current Working Directory: C:\Users\Akhil\Downloads\project\TradeX_raw\TTcon


In [5]:
files = os.listdir(path)
file1 = [f for f in files if f.startswith("tata_dialer_call_report_")][0]
file2 = [f for f in files if f.startswith("call_report_")][0]

In [6]:
T1 = pd.read_csv(os.path.join(path, file1))
T2 = pd.read_csv(os.path.join(path, file2))
T2 = T2.dropna(subset=["Call Flow"])
T2 = T2.rename(columns={"Client Number": "Customer Number", "Status": "Call Status"})
T2["Connected to Agent"] = (
    T2["Call Flow"]
    .str.findall(r'Agent:\s*([^(,]+)')
    .str[0]
    .str.strip()
)
T2["Call Start Date"] = T2["Call Flow"].str.extract(r'\((\d{2}-\d{2}-\d{4})\s')
T2["Call Start Time"] = T2["Call Flow"].str.extract(r'\d{4}\s(\d{2}:\d{2}:\d{2})')
T2 = T2.dropna(subset=["Call Start Date"])
T2["Customer Number"] = T2["Call Flow"].str.extract(r'PJSIP/(\+?\d+)@')
T2["Answer Duration (HH:MM:SS)"] = ( pd.to_timedelta(T2["Outbound Duration"].astype(float), unit="s") .astype(str) .str[-8:] .replace("NaT", "00:00:00") )
T2["Total Call Duration (HH:MM:SS)"] = ( pd.to_timedelta(T2["Call Duration"].astype(float), unit="s") .astype(str) .str[-8:] .replace("NaT", "00:00:00") )
T2["Hold Duration (HH:MM:SS)"] = ( pd.Series([pd.to_timedelta(0, unit="s")] * len(T2)) .dt.total_seconds() .astype(int) .apply(lambda x: f"{x//3600:02d}:{(x%3600)//60:02d}:{x%60:02d}") )
T2["Call Start Date"] = pd.to_datetime(T2["Call Start Date"], format='%d-%m-%Y') 
T2["Call Start Date"] = T2["Call Start Date"].dt.strftime('%Y-%m-%d')
T2 = T2[['Call Start Date', 'Connected to Agent', 'Customer Number', 'Call Status', 'Answer Duration (HH:MM:SS)','Hold Duration (HH:MM:SS)',  'Total Call Duration (HH:MM:SS)', 'Call Start Time']]
T1 = T1[['Call Start Date', 'Connected to Agent', 'Customer Number', 'Call Status', 'Answer Duration (HH:MM:SS)','Hold Duration (HH:MM:SS)',  'Total Call Duration (HH:MM:SS)', 'Call Start Time']]
T_total = pd.concat([T1, T2], ignore_index=True)
yesterday = datetime.today() - timedelta(days=1)

In [7]:
T_total[ T_total['Connected to Agent'] == '580 Riddhi (Extension-0605454200214)' ]

,Call Start Date,Connected to Agent,Customer Number,Call Status,Answer Duration (HH:MM:SS),Hold Duration (HH:MM:SS),Total Call Duration (HH:MM:SS),Call Start Time
6,2026-04-03,580 Riddhi (Extension-0605454200214),917973975476,Answered,00:01:48,00:00:00,00:02:00,19:41:42
8,2026-04-03,580 Riddhi (Extension-0605454200214),917769837453,Missed,00:00:00,00:00:00,00:00:06,19:41:25
10,2026-04-03,580 Riddhi (Extension-0605454200214),919584249106,Missed,00:00:00,00:00:00,00:00:10,19:41:00
12,2026-04-03,580 Riddhi (Extension-0605454200214),919832795124,Missed,00:00:00,00:00:00,00:00:31,19:40:17
22,2026-04-03,580 Riddhi (Extension-0605454200214),919864891853,Answered,00:00:47,00:00:00,00:01:00,19:37:38
...,...,...,...,...,...,...,...,...
6049,2026-04-03,580 Riddhi (Extension-0605454200214),919058448344,Answered,00:07:54,00:00:00,00:08:05,09:53:11
6071,2026-04-03,580 Riddhi (Extension-0605454200214),919634687571,Missed,00:00:00,00:00:00,00:00:13,09:50:23
6091,2026-04-03,580 Riddhi (Extension-0605454200214),917490053711,Missed,00:00:00,00:00:00,00:00:31,09:47:07
6101,2026-04-03,580 Riddhi (Extension-0605454200214),918805539018,Missed,00:00:00,00:00:00,00:00:21,09:45:59


In [8]:
file_name = f"T_{yesterday.day:02d}_{yesterday.month:02d}.csv"

In [9]:
output_path = os.path.join(r"C:\Users\Akhil\Downloads\project\TradeX_raw", file_name) 
T_total.to_csv(output_path, index=False)
T1.head(2)
T2.head(2)
path = r"C:\Users\Akhil\Downloads\project\TradeX_raw\TTcon"

In [10]:
path = r"C:\Users\Akhil\Downloads\project\TradeX_raw"

#\pyton automation\Tradex_dialer_raw
# Change the working directory
os.chdir(path)

# Confirm the new working directory
print("Current Working Directory:", os.getcwd())

Current Working Directory: C:\Users\Akhil\Downloads\project\TradeX_raw


## Adjustment

In [2073]:
path = r'C:\Users\Akhil\Downloads\project\TradeX_raw'
os.chdir(path)

In [2074]:
# voiso = pd.read_csv('VT_12-16.csv',low_memory=False)
tata = T_total.copy()
know = pd.read_csv('K_03_04.csv',low_memory=False)
Qconn = pd.read_csv('Q_18_07.csv',low_memory=False)
stringee = stringee.copy()

## Check_point_1

In [2075]:
# print("voiso:", voiso['Date and time'].unique()[:5])
print("tata:", tata['Call Start Date'].unique()[:5])
print("know:", know['Date and Time'].unique()[:5])
print("Qconn:", Qconn['Date time'].unique()[:5])
print("stringee:", stringee['Start time'].unique()[:5])

tata: ['2026-04-03']
know: ['2026-04-03 20:56:35' '2026-04-03 20:56:27' '2026-04-03 19:07:08'
 '2026-04-03 19:03:05' '2026-04-03 19:02:34']
Qconn: []
stringee: <DatetimeArray>
['2026-04-03 22:03:57.385000', '2026-04-03 21:51:17.715000',
 '2026-04-03 21:09:37.750000', '2026-04-03 20:57:36.698000',
 '2026-04-03 20:13:03.597000']
Length: 5, dtype: datetime64[ns]


In [2076]:
stringee

,ID,Customer number,Hotline,Call type,Start time,End time,Queue duration,Answer duration,Account,Hold duration,Contact,Company ID,End of Call code,Call status
0,call-vn-1-9EYTIVYVSP-1772161806023,+918292598301,917949152736,Inbound calls,2026-04-03 22:03:57.385,2026-04-03 22:03:57.398,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN
1,call-vn-1-9EYTIVYVSP-1772161805691,+916352157037,917949152736,Inbound calls,2026-04-03 21:51:17.715,2026-04-03 21:51:17.725,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN
2,call-vn-1-9EYTIVYVSP-1772161804430,+919934865745,917949152736,Inbound calls,2026-04-03 21:09:37.750,2026-04-03 21:09:37.759,00:00:00,00:00:00,NaN,00:00:00,Contact 919934865745,NaN,USER_END_CALL,NaN
3,call-vn-1-9EYTIVYVSP-1772161803949,+919877693953,917949152736,Inbound calls,2026-04-03 20:57:36.698,2026-04-03 20:57:36.710,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN
4,call-vn-1-9EYTIVYVSP-1772161800675,+918279984337,917949152736,Inbound calls,2026-04-03 20:13:03.597,2026-04-03 20:13:03.607,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,call-vn-1-9EYTIVYVSP-1772161388085,919177092480,917949152736,Outbound calls,2026-04-03 09:19:07.770,2026-04-03 09:19:48.726,00:00:40,00:00:00,esha@bluewave.com,00:00:35,NaN,NaN,USER_END_CALL,NaN
2498,call-vn-1-9EYTIVYVSP-1772161387818,918010543210,917949152736,Outbound calls,2026-04-03 09:18:28.324,2026-04-03 09:18:48.733,00:00:20,00:00:00,esha@bluewave.com,00:00:00,NaN,NaN,480 Temporarily Unavailable,NaN
2499,call-vn-1-9EYTIVYVSP-1772161387293,919248792487,917949152736,Outbound calls,2026-04-03 09:17:09.257,2026-04-03 09:18:09.194,00:00:59,00:00:00,esha@bluewave.com,00:00:00,NaN,NaN,USER_END_CALL,NaN
2500,call-vn-1-9EYTIVYVSP-1772161387117,917032320554,917949152736,Outbound calls,2026-04-03 09:16:38.985,2026-04-03 09:17:03.925,00:00:24,00:00:00,esha@bluewave.com,00:00:00,NaN,NaN,480 Temporarily Unavailable,NaN


In [2078]:
stringee.columns

Index(['ID', 'Customer number', 'Hotline', 'Call type', 'Start time',
       'End time', 'Queue duration', 'Answer duration', 'Account',
       'Hold duration', 'Contact', 'Company ID', 'End of Call code',
       'Call status'],
      dtype='object')

In [2079]:
stringee['Answer duration'].dtype

dtype('O')

In [2080]:
# Function to convert durations in hh:mm:ss format to timedelta
def duration_to_timedelta(duration):
    hours, minutes, seconds = map(int, duration.split(':'))
    return timedelta(hours=hours, minutes=minutes, seconds=seconds)

# Convert the 'Queue duration' and 'Answer duration' columns to timedelta
stringee['Queue Duration (timedelta)'] = stringee['Queue duration'].apply(duration_to_timedelta)
stringee['Answer Duration (timedelta)'] = stringee['Answer duration'].apply(duration_to_timedelta)

# Calculate the total duration as timedelta
stringee['Total Duration (timedelta)'] = stringee['Queue Duration (timedelta)'] + stringee['Answer Duration (timedelta)']

# Convert the total duration back to hh:mm:ss format (remove "0 days")
stringee['Total Duration'] = stringee['Total Duration (timedelta)'].apply(
    lambda x: str(x).split("days")[-1].strip()
)

# Drop intermediate timedelta columns if not needed
stringee = stringee.drop(columns=['Queue Duration (timedelta)', 'Answer Duration (timedelta)', 'Total Duration (timedelta)'])


In [2081]:
stringee

,ID,Customer number,Hotline,Call type,Start time,End time,Queue duration,Answer duration,Account,Hold duration,Contact,Company ID,End of Call code,Call status,Total Duration
0,call-vn-1-9EYTIVYVSP-1772161806023,+918292598301,917949152736,Inbound calls,2026-04-03 23:33:57.385,2026-04-03 22:03:57.398,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN,00:00:00
1,call-vn-1-9EYTIVYVSP-1772161805691,+916352157037,917949152736,Inbound calls,2026-04-03 23:21:17.715,2026-04-03 21:51:17.725,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN,00:00:00
2,call-vn-1-9EYTIVYVSP-1772161804430,+919934865745,917949152736,Inbound calls,2026-04-03 22:39:37.750,2026-04-03 21:09:37.759,00:00:00,00:00:00,NaN,00:00:00,Contact 919934865745,NaN,USER_END_CALL,NaN,00:00:00
3,call-vn-1-9EYTIVYVSP-1772161803949,+919877693953,917949152736,Inbound calls,2026-04-03 22:27:36.698,2026-04-03 20:57:36.710,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN,00:00:00
4,call-vn-1-9EYTIVYVSP-1772161800675,+918279984337,917949152736,Inbound calls,2026-04-03 21:43:03.597,2026-04-03 20:13:03.607,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,NaN,00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,call-vn-1-9EYTIVYVSP-1772161388085,919177092480,917949152736,Outbound calls,2026-04-03 10:49:07.770,2026-04-03 09:19:48.726,00:00:40,00:00:00,esha@bluewave.com,00:00:35,NaN,NaN,USER_END_CALL,NaN,00:00:40
2498,call-vn-1-9EYTIVYVSP-1772161387818,918010543210,917949152736,Outbound calls,2026-04-03 10:48:28.324,2026-04-03 09:18:48.733,00:00:20,00:00:00,esha@bluewave.com,00:00:00,NaN,NaN,480 Temporarily Unavailable,NaN,00:00:20
2499,call-vn-1-9EYTIVYVSP-1772161387293,919248792487,917949152736,Outbound calls,2026-04-03 10:47:09.257,2026-04-03 09:18:09.194,00:00:59,00:00:00,esha@bluewave.com,00:00:00,NaN,NaN,USER_END_CALL,NaN,00:00:59
2500,call-vn-1-9EYTIVYVSP-1772161387117,917032320554,917949152736,Outbound calls,2026-04-03 10:46:38.985,2026-04-03 09:17:03.925,00:00:24,00:00:00,esha@bluewave.com,00:00:00,NaN,NaN,480 Temporarily Unavailable,NaN,00:00:24


## ETL

In [2082]:
new_tata = tata[['Call Start Date', 'Connected to Agent','Call Status','Answer Duration (HH:MM:SS)',
                 'Hold Duration (HH:MM:SS)','Total Call Duration (HH:MM:SS)','Call Start Time','Customer Number']]

In [ ]:
new_tata[ new_tata['Connected to Agent'] == '580 Riddhi (Extension-0605454200214)' ]

In [2083]:

new_Know = know[['Date and Time', 'Agent Name','Call Status', 'Talk Time (hh:mm:ss)', 'Hold Time (hh:mm:ss)','Total Call Duration (hh:mm:ss)','Customer']]

In [2084]:
# new_voiso =  voiso[['Date and time','Agent(s)','Disposition','Talk time','Duration','DNIS/To']]

In [2085]:
new_qconn = Qconn[['Date time','Agent Mobile','Call Event','Transfer Duration','Duration','User Mobile']]

In [2086]:
new_string = stringee[['Start time','Account','Call status','Answer duration','Hold duration','Total Duration','Customer number']]

## Voiso date 

In [2087]:
# new_voiso['Date and time'].unique()

In [2088]:
# # Function to convert 'Date and time' from mm/dd/yyyy HH:mm:ss to yyyy/mm/dd HH:mm:ss
# def fix_voiso_datetime(df, col_name):
#     # Convert using pd.to_datetime, specify the format as mm/dd/yyyy
#     df[col_name] = pd.to_datetime(df[col_name], errors='coerce', format='%m/%d/%Y %H:%M:%S') #
    
#     # Format the datetime to 'yyyy/mm/dd HH:mm:ss'
#     df[col_name] = df[col_name].dt.strftime('%Y/%m/%d %H:%M:%S')
    
#     return df

# # Apply the function to 'Date and time' column in Voiso
# new_voiso = fix_voiso_datetime(new_voiso, 'Date and time')

# # Verify the result
# print("Voiso Date and time after fixing format:", new_voiso['Date and time'].unique())


## string date format

In [2089]:
#####check
new_string['Start time'].head()

0   2026-04-03 23:33:57.385
1   2026-04-03 23:21:17.715
2   2026-04-03 22:39:37.750
3   2026-04-03 22:27:36.698
4   2026-04-03 21:43:03.597
Name: Start time, dtype: datetime64[ns]

In [2090]:
# Function to fix 'new_string' Date Time format
def fix_string_datetime(date_str):
    try:
        # Parse the date and time, assuming format mm/dd/yyyy hh:mm:ss AM/PM
        parsed_date = pd.to_datetime(date_str, errors='coerce', format='%Y-%m-%d %H:%M:%S.%f')
        return parsed_date
    except Exception as e:
        return pd.NaT  # Return NaT for any unparseable date

# Apply this function to the 'Date and Time' column in 'new_string'
new_string['Start time'] = new_string['Start time'].apply(fix_string_datetime)

# Now format it to yyyy/mm/dd HH:mm:ss
new_string['Start time'] = new_string['Start time'].dt.strftime('%Y/%m/%d %H:%M:%S')

# Check the result
print("String unique date formats after fixing:", new_string['Start time'].head())

String unique date formats after fixing: 0    2026/04/03 23:33:57
1    2026/04/03 23:21:17
2    2026/04/03 22:39:37
3    2026/04/03 22:27:36
4    2026/04/03 21:43:03
Name: Start time, dtype: object


C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\1218228788.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_string['Start time'] = new_string['Start time'].apply(fix_string_datetime)
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\1218228788.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_string['Start time'] = new_string['Start time'].dt.strftime('%Y/%m/%d %H:%M:%S')


In [2091]:
new_Know['Date and Time'].head()


0    2026-04-03 20:56:35
1    2026-04-03 20:56:27
2    2026-04-03 19:07:08
3    2026-04-03 19:03:05
4    2026-04-03 19:02:34
Name: Date and Time, dtype: object

In [2092]:
new_qconn['Date time'].head()

Series([], Name: Date time, dtype: object)

In [2093]:
# Convert 'Date and Time' column in new_Know to datetime format
new_qconn['Date time'] = pd.to_datetime(new_qconn['Date time'], errors='coerce', format='%Y-%m-%d %H:%M:%S')
new_Know['Date and Time'] = pd.to_datetime(new_Know['Date and Time'], errors='coerce', format='%Y-%m-%d %H:%M:%S') #%d/%m/%Y

# Now check the type of 'Date' again in both dataframes
print("Data type of 'Date' in new_qconn after conversion:", new_qconn['Date time'].dtype)
print("Data type of 'Date' in new_Know after conversion:", new_Know['Date and Time'].dtype)

# Extract date and call start time for new_Know
new_Know['Date'] = new_Know['Date and Time'].dt.date  # Extract date part
new_Know['Call Start Time'] = new_Know['Date and Time'].dt.strftime('%H:%M:%S')  # Extract time part

# Extract date and call start time for new_qconn (if you haven't done this yet)
new_qconn['Date'] = new_qconn['Date time'].dt.date  # Extract date part
new_qconn['Call Start Time'] = new_qconn['Date time'].dt.strftime('%H:%M:%S')  # Extract time part

# Check the resulting dataframes to ensure extraction is successful
print(new_qconn[['Date', 'Call Start Time']].head())
print(new_Know[['Date', 'Call Start Time']].head())


Data type of 'Date' in new_qconn after conversion: datetime64[ns]
Data type of 'Date' in new_Know after conversion: datetime64[ns]
Empty DataFrame
Columns: [Date, Call Start Time]
Index: []
         Date Call Start Time
0  2026-04-03        20:56:35
1  2026-04-03        20:56:27
2  2026-04-03        19:07:08
3  2026-04-03        19:03:05
4  2026-04-03        19:02:34


C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\4068174409.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_Know['Date and Time'] = pd.to_datetime(new_Know['Date and Time'], errors='coerce', format='%Y-%m-%d %H:%M:%S') #%d/%m/%Y
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\4068174409.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_Know['Date'] = new_Know['Date and Time'].dt.date  # Extract date part
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\4068174409.py:11: SettingWithCopyWar

In [2094]:
# Find rows where 'Date time' is NaT
invalid_dates = new_qconn[new_qconn['Date time'].isna()]
print(invalid_dates)


Empty DataFrame
Columns: [Date time, Agent Mobile, Call Event, Transfer Duration, Duration, User Mobile, Date, Call Start Time]
Index: []


In [2095]:
(new_qconn['Date'].unique())

array([], dtype=object)

In [2096]:
# Check the data type of the 'Date' column
print("Data type of 'Date' in new_qconn:", new_qconn['Date time'].dtype)
print("Data type of 'Date' in new_Know:", new_Know['Date and Time'].dtype)


Data type of 'Date' in new_qconn: datetime64[ns]
Data type of 'Date' in new_Know: datetime64[ns]


In [2097]:
# Function to separate Date and Call Start Time
def split_date_time(df, col_name):
    # Convert to datetime first (if not already done)
    df[col_name] = pd.to_datetime(df[col_name], errors='coerce', format='%Y/%m/%d %H:%M:%S')

    # Extract the Date (yyyy/mm/dd) and Time (hh:mm:ss)
    df['Date'] = df[col_name].dt.date
    df['Call Start Time'] = df[col_name].dt.strftime('%H:%M:%S')

    return df

# Apply the function to each dataframe
# new_voiso = split_date_time(new_voiso, 'Date and time')
new_string = split_date_time(new_string, 'Start time')

# Verify the changes
print(new_qconn[['Date', 'Call Start Time']].head())
# print(new_voiso[['Date', 'Call Start Time']].head())
print(new_Know[['Date', 'Call Start Time']].head())
print(new_string[['Date', 'Call Start Time']].head())


Empty DataFrame
Columns: [Date, Call Start Time]
Index: []
         Date Call Start Time
0  2026-04-03        20:56:35
1  2026-04-03        20:56:27
2  2026-04-03        19:07:08
3  2026-04-03        19:03:05
4  2026-04-03        19:02:34
         Date Call Start Time
0  2026-04-03        23:33:57
1  2026-04-03        23:21:17
2  2026-04-03        22:39:37
3  2026-04-03        22:27:36
4  2026-04-03        21:43:03


C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\1294338338.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col_name] = pd.to_datetime(df[col_name], errors='coerce', format='%Y/%m/%d %H:%M:%S')
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\1294338338.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = df[col_name].dt.date
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\1294338338.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [2098]:
new_qconn

,Date time,Agent Mobile,Call Event,Transfer Duration,Duration,User Mobile,Date,Call Start Time


### copy of main dfs

In [2099]:
tata_copy = new_tata.copy()
know_copy = new_Know.copy()
# voiso_copy = new_voiso.copy()
qconn_copy = new_qconn.copy()
string_copy = new_string.copy()

### insert source 

In [2100]:
tata_copy['Source'] = 'Tata'
know_copy['Source'] = 'Knowlarity'
# voiso_copy['Source'] = 'Voiso'
qconn_copy['Source'] = 'Qkonnect'
string_copy['Source'] = 'Stringee'


In [2101]:
print(tata_copy.dtypes)
print(string_copy['Date'].unique())

Call Start Date                   object
Connected to Agent                object
Call Status                       object
Answer Duration (HH:MM:SS)        object
Hold Duration (HH:MM:SS)          object
Total Call Duration (HH:MM:SS)    object
Call Start Time                   object
Customer Number                   object
Source                            object
dtype: object
[datetime.date(2026, 4, 3)]


### Rename Columns

In [2102]:
tata_copy.rename(columns={
    'Call Start Date': 'Date',
    'Connected to Agent': 'Dialer Name',
    'Customer Number' : 'Number',
    'Call Status': 'Call Status',
    'Answer Duration (HH:MM:SS)': 'Talk Time',
    'Hold Duration (HH:MM:SS)': 'Hold Time',
    'Total Call Duration (HH:MM:SS)': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)

know_copy.rename(columns={
    'Date': 'Date',
    'Agent Name': 'Dialer Name',
    'Customer': 'Number',
    'Call Status': 'Call Status',
    'Talk Time (hh:mm:ss)': 'Talk Time',
    'Hold Time (hh:mm:ss)': 'Hold Time',
    'Total Call Duration (hh:mm:ss)': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)

# voiso_copy.rename(columns={
#     'Date': 'Date',
#     'Agent(s)': 'Dialer Name',
#     'DNIS/To': 'Number',
#     'Disposition': 'Call Status',
#     'Talk time': 'Talk Time',
#     'Duration': 'Total Call Duration',
#     'Call Start Time': 'Call Start Time'
# }, inplace=True)

qconn_copy.rename(columns={
    'Date': 'Date',
    'Agent Mobile': 'Dialer Name',
    'User Mobile': 'Number',
    'Call Event': 'Call Status',
    'Transfer Duration': 'Talk Time',
    'Duration': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)

string_copy.rename(columns={
    'Date': 'Date',    
    'Account': 'Dialer Name',
    'Customer number': 'Number',
    'Call status': 'Call Status',
    'Answer duration': 'Talk Time',
    'Hold duration': 'Hold Time',
    'Total Duration': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)



In [2103]:
tata_copy[ tata_copy['Dialer Name'] == '580 Riddhi' ]

,Date,Dialer Name,Call Status,Talk Time,Hold Time,Total Call Duration,Call Start Time,Number,Source


In [1947]:
print(qconn_copy.shape)
# print(voiso_copy.shape)
print(know_copy.shape)  
print(tata_copy.shape)
print(string_copy.shape)

(0, 9)
(1128, 10)
(13113, 9)
(2475, 10)


In [1948]:
print(string_copy['Date'].unique())
print(tata_copy['Date'].unique())
print(know_copy['Date'].unique())
# print(voiso_copy['Date'].unique())
print(qconn_copy['Date'].unique())


[datetime.date(2026, 4, 3)]
['2026-04-03']
[datetime.date(2026, 4, 3)]
[]


In [1949]:
tata_copy.shape, know_copy.shape, qconn_copy.shape,string_copy.shape

((13113, 9), (1128, 10), (0, 9), (2475, 10))

In [1950]:

# Select only the required columns from each dataframe
tata_selected = tata_copy[[ 'Source','Date', 'Dialer Name','Number', 'Call Status','Call Start Time','Total Call Duration', 'Talk Time', 'Hold Time']]
know_selected = know_copy[[ 'Source','Date', 'Dialer Name','Number' ,'Call Status', 'Call Start Time','Total Call Duration','Talk Time', 'Hold Time']]
# voiso_selected = voiso_copy[[ 'Source','Date', 'Dialer Name', 'Number','Call Status','Call Start Time','Total Call Duration', 'Talk Time']]
qconn_selected = qconn_copy[['Source','Date', 'Dialer Name','Number', 'Call Status','Call Start Time','Total Call Duration', 'Talk Time']]
string_selected = string_copy[['Source','Date', 'Dialer Name','Number', 'Call Status','Call Start Time','Total Call Duration', 'Talk Time', 'Hold Time']]


# Now concatenate
combined = pd.concat([tata_selected, know_selected, qconn_selected, string_selected], ignore_index=True)

In [1951]:
combined[ combined['Dialer Name'] == '580 Riddhi' ]

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time


In [1952]:
combined

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh (Extension-0605454200250),918918257549,Missed,19:44:43,00:00:07,00:00:00,00:00:00
1,Tata,2026-04-03,597 Suresh (Extension-0605454200250),919727948999,Missed,19:44:16,00:00:05,00:00:00,00:00:00
2,Tata,2026-04-03,597 Suresh (Extension-0605454200250),918367656635,Missed,19:43:45,00:00:07,00:00:00,00:00:00
3,Tata,2026-04-03,597 Suresh (Extension-0605454200250),918303691062,Missed,19:43:21,00:00:05,00:00:00,00:00:00
4,Tata,2026-04-03,597 Suresh (Extension-0605454200250),919864480486,Missed,19:42:51,00:00:09,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
16711,Stringee,2026-04-03,esha@bluewave.com,919177092480,NaN,10:49:07,00:00:40,00:00:00,00:00:35
16712,Stringee,2026-04-03,esha@bluewave.com,918010543210,NaN,10:48:28,00:00:20,00:00:00,00:00:00
16713,Stringee,2026-04-03,esha@bluewave.com,919248792487,NaN,10:47:09,00:00:59,00:00:00,00:00:00
16714,Stringee,2026-04-03,esha@bluewave.com,917032320554,NaN,10:46:38,00:00:24,00:00:00,00:00:00


In [1953]:
print(combined.isnull().sum())
print(combined.shape)

Source                    0
Date                      0
Dialer Name             180
Number                   11
Call Status            2475
Call Start Time           0
Total Call Duration       0
Talk Time                 0
Hold Time               232
dtype: int64
(16716, 9)


## Check_point_2

In [1954]:
# Filter rows where Date is null
null_date_entries = combined[combined['Date'].isnull()]

# Display the filtered DataFrame
null_date_entries


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time


In [1955]:
combined['Date'] = pd.to_datetime(combined['Date'])

In [1956]:
combined['Total Call Duration'] = pd.to_timedelta(
    combined['Total Call Duration'].astype(str)
)

In [1957]:
combined['Total Call Duration'].unique()

<TimedeltaArray>
['0 days 00:00:07', '0 days 00:00:05', '0 days 00:00:09', '0 days 00:00:08',
 '0 days 00:02:00', '0 days 00:00:06', '0 days 00:00:10', '0 days 00:00:31',
 '0 days 00:00:33', '0 days 00:00:16',
 ...
 '0 days 00:28:40', '0 days 00:03:13', '0 days 00:16:00', '0 days 00:02:58',
 '0 days 00:07:04', '0 days 00:04:32', '0 days 00:13:23', '0 days 00:14:31',
 '0 days 00:04:41', '0 days 00:10:51']
Length: 474, dtype: timedelta64[ns]

In [1958]:
combined_df =combined.copy()

In [1959]:
# Refine the regex to only remove unwanted patterns
combined_df['Dialer Name'] = combined_df['Dialer Name'].str.replace(r"\s*\([^)]*\)|@.*|;.*", "", regex=True)

# Fill missing 'Dialer Name' values with their original values if they were numeric
combined_df['Dialer Name'] = combined_df['Dialer Name'].fillna(combined['Dialer Name'])

In [1960]:
combined['Total Duration'] = combined['Total Call Duration'].apply(
    lambda x: str(x).split("days")[-1].strip()
)

In [1961]:
# Replace NaN in 'Hold time' with '00:00:00'
combined_df['Hold Time'] = combined_df['Hold Time'].fillna('00:00:00')

combined_df

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,Missed,19:44:43,0 days 00:00:07,00:00:00,00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,Missed,19:44:16,0 days 00:00:05,00:00:00,00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,Missed,19:43:45,0 days 00:00:07,00:00:00,00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,Missed,19:43:21,0 days 00:00:05,00:00:00,00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,Missed,19:42:51,0 days 00:00:09,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
16711,Stringee,2026-04-03,esha,919177092480,NaN,10:49:07,0 days 00:00:40,00:00:00,00:00:35
16712,Stringee,2026-04-03,esha,918010543210,NaN,10:48:28,0 days 00:00:20,00:00:00,00:00:00
16713,Stringee,2026-04-03,esha,919248792487,NaN,10:47:09,0 days 00:00:59,00:00:00,00:00:00
16714,Stringee,2026-04-03,esha,917032320554,NaN,10:46:38,0 days 00:00:24,00:00:00,00:00:00


In [1962]:
A = combined_df.copy()

In [1963]:
# Remove rows where 'Dialer Name' is null
A1 = A[A['Dialer Name'].notnull()]

# Verify the result
print(f"Number of rows after removing null 'Dialer Name': {len(A1)}")

Number of rows after removing null 'Dialer Name': 16536


In [1964]:
A1[A1['Dialer Name'].str.contains('223', na=False)]['Dialer Name'].unique()

array(['223 abhishek-Extension'], dtype=object)

In [1965]:
A1.describe()

,Date,Total Call Duration
count,16536,16536
mean,2026-04-03 00:00:00,0 days 00:00:39.340287856
min,2026-04-03 00:00:00,0 days 00:00:00
25%,2026-04-03 00:00:00,0 days 00:00:14
50%,2026-04-03 00:00:00,0 days 00:00:26
75%,2026-04-03 00:00:00,0 days 00:00:33
max,2026-04-03 00:00:00,0 days 00:41:16
std,NaN,0 days 00:01:38.876424571


In [1966]:
# Check for null values in the 'Dialer Name' column
null_dialer_name_count = combined_df['Dialer Name'].isnull().sum()

# Print the result
print(f"Number of null values in 'Dialer Name': {null_dialer_name_count}")

Number of null values in 'Dialer Name': 180


In [1967]:
A1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,Missed,19:44:43,0 days 00:00:07,00:00:00,00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,Missed,19:44:16,0 days 00:00:05,00:00:00,00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,Missed,19:43:45,0 days 00:00:07,00:00:00,00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,Missed,19:43:21,0 days 00:00:05,00:00:00,00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,Missed,19:42:51,0 days 00:00:09,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
16711,Stringee,2026-04-03,esha,919177092480,NaN,10:49:07,0 days 00:00:40,00:00:00,00:00:35
16712,Stringee,2026-04-03,esha,918010543210,NaN,10:48:28,0 days 00:00:20,00:00:00,00:00:00
16713,Stringee,2026-04-03,esha,919248792487,NaN,10:47:09,0 days 00:00:59,00:00:00,00:00:00
16714,Stringee,2026-04-03,esha,917032320554,NaN,10:46:38,0 days 00:00:24,00:00:00,00:00:00


## Check_point_3

In [1968]:
unique_dates_per_source = A1.groupby('Source')['Date'].unique()
print(unique_dates_per_source)

Source
Knowlarity    [2026-04-03 00:00:00]
Stringee      [2026-04-03 00:00:00]
Tata          [2026-04-03 00:00:00]
Name: Date, dtype: object


In [1969]:
A1.dtypes

Source                          object
Date                    datetime64[ns]
Dialer Name                     object
Number                          object
Call Status                     object
Call Start Time                 object
Total Call Duration    timedelta64[ns]
Talk Time                       object
Hold Time                       object
dtype: object

In [1970]:
duration_cols = ['Talk Time', 'Hold Time', 'Total Call Duration']
for col in duration_cols:
    A1[col] = pd.to_timedelta(
        A1[col].astype(str).str.strip(),
        errors='coerce'
    )

C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\2864105437.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1[col] = pd.to_timedelta(
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\2864105437.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1[col] = pd.to_timedelta(
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\2864105437.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats 

In [1971]:
for col in duration_cols:
    A1[col] = pd.to_timedelta(
        A1[col],
        errors='coerce'
    )


C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\749567782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1[col] = pd.to_timedelta(


In [1972]:
A1.dtypes

Source                          object
Date                    datetime64[ns]
Dialer Name                     object
Number                          object
Call Status                     object
Call Start Time                 object
Total Call Duration    timedelta64[ns]
Talk Time              timedelta64[ns]
Hold Time              timedelta64[ns]
dtype: object

In [1973]:
import re

def normalize_talk_time(talk_time):
    # If the value is in seconds (only digits), convert it to hh:mm:ss
    if re.match(r"^\d+$", str(talk_time)):
        seconds = int(talk_time)
        hours = seconds // 3600
        minutes = (seconds % 3600) // 60
        seconds = seconds % 60
        return f"{hours:02}:{minutes:02}:{seconds:02}"
    elif re.match(r"^\d+:\d+:\d+$", str(talk_time)):
        parts = talk_time.split(":")
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return f"{hours:02}:{minutes:02}:{seconds:02}"
    else:
        # Return the value as-is if it doesn't match expected formats
        return talk_time

# Apply the normalization function to the 'Talk Time' column
A1['Talk Time'] = A1['Talk Time'].apply(normalize_talk_time)

# Apply the normalization function to the 'Total Call Duration' column
A1['Total Call Duration'] = A1['Total Call Duration'].apply(normalize_talk_time)


print(A1[['Talk Time', 'Total Call Duration']].head())

  Talk Time Total Call Duration
0    0 days     0 days 00:00:07
1    0 days     0 days 00:00:05
2    0 days     0 days 00:00:07
3    0 days     0 days 00:00:05
4    0 days     0 days 00:00:09


C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\4084048447.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1['Talk Time'] = A1['Talk Time'].apply(normalize_talk_time)
C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\4084048447.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1['Total Call Duration'] = A1['Total Call Duration'].apply(normalize_talk_time)


In [1974]:
unique_talk_time_formats = A1['Talk Time'].unique()

print(f"Unique formats in 'Talk Time': {unique_talk_time_formats}")

Unique formats in 'Talk Time': <TimedeltaArray>
['0 days 00:00:00', '0 days 00:01:48', '0 days 00:00:21', '0 days 00:00:47',
 '0 days 00:00:19', '0 days 00:01:01', '0 days 00:00:02', '0 days 00:01:18',
 '0 days 00:00:49', '0 days 00:00:03',
 ...
 '0 days 00:09:57', '0 days 00:15:16', '0 days 00:06:31', '0 days 00:28:23',
 '0 days 00:03:02', '0 days 00:15:53', '0 days 00:04:24', '0 days 00:13:15',
 '0 days 00:14:01', '0 days 00:10:26']
Length: 462, dtype: timedelta64[ns]


#### Unique call status (AOI)

In [1975]:
A1['Call Status'].unique()

array(['Missed', 'Answered', 'Dropped', nan], dtype=object)

In [1976]:
A1['Call Status'] = A1['Call Status'].str.lower().apply(
    lambda x: 'connected' if x == 'answered' else 'not connected'
)

C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\903979383.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1['Call Status'] = A1['Call Status'].str.lower().apply(


In [1977]:
A1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...
16711,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35
16712,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00
16713,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00
16714,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00


In [1978]:
A1['Call Status'].unique()

array(['not connected', 'connected'], dtype=object)

In [1979]:
len(A1['Dialer Name'].unique())

77

In [1980]:
A1['Dialer Name'].unique()

array(['597 Suresh', '580 Riddhi', '577 Saurabh', '596 Shambu',
       '475 Sushil', '502 Fahrukh', '600 Yatin', 'reeya', '---', 'mohit',
       '265 Amrut', '514 Tarun', '612 Sandeep', '298 Sujal', 'musa',
       'karan', '210 Sankalp', '264  Lakshmi', '259 Khushboo',
       '214 Shubham', 'zaid', '258 shivansh', 'Rajinder', '539 Divya',
       '599 Aamir', 'alina', 'siddhesh', '88 Waman', '614 Irfan',
       '251 Chandan-Extension', 'shiv-Extension', 'alina-Extension',
       'haresh-Extension', '614 Irfan-Extension', '498 Ankita-Extension',
       'adharv-Extension', '223 abhishek-Extension', 'shahid-Extension',
       '520 Nair-Extension', 'aaryan-Extension', 'sana-Extension',
       '580 Riddhi-Extension', 'reeya-Extension', 'vaibhav-Extension',
       '88 Waman-Extension', '569 Maaz-Extension', '497 Anjali-Extension',
       'kanika-Extension', '537 Mitali-Extension', 'ritesh-Extension',
       'amrit-Extension', 'madhur-Extension', '584 Sneha-Extension',
       'deepa-Extension'

In [1981]:
# Count the occurrences of each unique 'Dialer Name' and sort from highest to lowest
dialer_name_counts = A1['Dialer Name'].value_counts().sort_values(ascending=False)

print(dialer_name_counts)

Dialer Name
---                     963
475 Sushil              359
537 Mitali-Extension    346
259 Khushboo            344
497 Anjali-Extension    335
                       ... 
alina                     5
88 Waman                  3
reeya                     2
614 Irfan                 1
siddhesh-Extension        1
Name: count, Length: 77, dtype: int64


In [1982]:

combined_df_1 = A1[~A1['Dialer Name'].isin([None, '---'])]

combined_df_1.reset_index(drop=True, inplace=True)

combined_df_1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...
15568,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35
15569,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00
15570,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00
15571,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00


In [1983]:
combined_df_1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...
15568,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35
15569,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00
15570,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00
15571,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00


In [1984]:
combined_df_1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...
15568,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35
15569,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00
15570,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00
15571,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00


In [1985]:
combined_df_1["Dialer Name"] = combined_df_1["Dialer Name"].str.replace("-Extension", "", regex=False)
combined_df_1

C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\1705523446.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_1["Dialer Name"] = combined_df_1["Dialer Name"].str.replace("-Extension", "", regex=False)


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...
15568,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35
15569,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00
15570,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00
15571,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00


In [1986]:
combined_df_1 = combined_df_1[combined_df_1['Date'] != '30/03/2026']
combined_df_1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-04-03,597 Suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00
1,Tata,2026-04-03,597 Suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00
2,Tata,2026-04-03,597 Suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00
3,Tata,2026-04-03,597 Suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00
4,Tata,2026-04-03,597 Suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...
15568,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35
15569,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00
15570,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00
15571,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00


In [1987]:
ref_d = pd.read_excel('Team_tradex.xlsx')

In [1988]:
ref_d.nunique()

Dialer Name      207
Dialer             6
Email            114
Employee code    114
Full Name        114
Pool              10
TL                 7
Vertical           1
dtype: int64

In [1989]:
ref= ref_d.copy()

In [1990]:
ref = ref[~ref['Email'].str.contains('inactive', case=False, na=False)]

In [1991]:
ref

,Dialer Name,Dialer,Email,Employee code,Full Name,Pool,TL,Vertical
0,223Abhishek,Knowlarity,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
1,223 Abhishek,Tata,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
3,412 Anuj,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit,Tata,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
...,...,...,...,...,...,...,...,...
202,Mohit,Tata,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,Alina,Tata,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,Siddhesh,Tata,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX
205,Reeya,Tata,reeya@nexora.live,E-2126,Reeya,OJT 1,Rehan,TradeX


## Checkpoint_4

In [1992]:
ref.nunique()

Dialer Name      207
Dialer             6
Email            114
Employee code    114
Full Name        114
Pool              10
TL                 7
Vertical           1
dtype: int64

In [1993]:
for df in [ref, combined_df_1]:
    df['Dialer Name'] = (
        df['Dialer Name']
        .astype(str)  # Convert all values to strings
        .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with a single space
        .str.strip()  # Remove leading and trailing spaces
        .str.replace(r'@.*', '', regex=True)  # Remove everything from and after '@'
        .str.replace(r'\(.*', '', regex=True)  # Remove everything from and after '('
        .str.strip()  # Remove any trailing spaces left after replacements
        .str.lower()  # Convert all text to lowercase
    )


In [1994]:
ref.shape

(207, 8)

In [1995]:
ref.isnull().sum()

Dialer Name      0
Dialer           0
Email            0
Employee code    0
Full Name        0
Pool             2
TL               2
Vertical         0
dtype: int64

In [1996]:
dup = ref[ref.duplicated(subset=['Dialer Name','Email','Dialer'], keep=False)]
dup

,Dialer Name,Dialer,Email,Employee code,Full Name,Pool,TL,Vertical


In [1997]:
ref = ref.drop_duplicates(subset=['Dialer Name','Email','Dialer'])

In [1998]:
ref.rename(columns={'Email': 'CRM ID'}, inplace=True)

In [1999]:
ref

,Dialer Name,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,223abhishek,Knowlarity,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
1,223 abhishek,Tata,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
3,412 anuj,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit,Tata,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
...,...,...,...,...,...,...,...,...
202,mohit,Tata,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,alina,Tata,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,siddhesh,Tata,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX
205,reeya,Tata,reeya@nexora.live,E-2126,Reeya,OJT 1,Rehan,TradeX


In [2000]:
# Ensure 'Dialer Name' in both dataframes is treated as a string
combined_df_1['Dialer Name'] = combined_df_1['Dialer Name'].astype(str)
ref['Dialer Name'] = ref['Dialer Name'].astype(str)

# Merge the dataframes on 'Dialer Name'
combined_df_2 = combined_df_1.merge(ref, how='left', left_on='Dialer Name', right_on='Dialer Name')


combined_df_2


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-04-03,597 suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
1,Tata,2026-04-03,597 suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
2,Tata,2026-04-03,597 suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
3,Tata,2026-04-03,597 suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
4,Tata,2026-04-03,597 suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16503,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16504,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16505,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16506,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX


In [2001]:
# Filter rows where 'CRM ID' is null
crm_id_null_df = combined_df_2[combined_df_2['CRM ID'].isnull()]


In [2002]:

crm_id_null_df['Dialer Name'].unique()

array([], dtype=object)

In [2003]:
crm_id_null_df

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical


## check_point_5

In [2004]:
source = crm_id_null_df.groupby('Source')['Dialer Name'].unique()
source

Series([], Name: Dialer Name, dtype: object)

In [2005]:
source

Series([], Name: Dialer Name, dtype: object)

In [2006]:
crm_id_null_df.shape

(0, 16)

## Raw data

In [2007]:
combined_df_2

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-04-03,597 suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
1,Tata,2026-04-03,597 suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
2,Tata,2026-04-03,597 suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
3,Tata,2026-04-03,597 suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
4,Tata,2026-04-03,597 suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16503,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16504,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16505,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16506,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX


In [2008]:
Dialers = combined_df_2[combined_df_2['CRM ID'].notnull() & combined_df_2['Talk Time'].notnull()].copy()

In [2009]:
Dialers.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 7
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
Dialer                 0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
dtype: int64

In [2010]:
Dialers = Dialers.drop_duplicates(subset=['Number','Call Start Time'])


In [2011]:
XXX = combined_df_2[combined_df_2['Date'].isnull()]
XXX

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical


In [2012]:
Dialers.dtypes

Source                          object
Date                    datetime64[ns]
Dialer Name                     object
Number                          object
Call Status                     object
Call Start Time                 object
Total Call Duration    timedelta64[ns]
Talk Time              timedelta64[ns]
Hold Time              timedelta64[ns]
Dialer                          object
CRM ID                          object
Employee code                   object
Full Name                       object
Pool                            object
TL                              object
Vertical                        object
dtype: object

In [2013]:
Dialers['Total Call Duration'].apply(type).value_counts()

Total Call Duration
<class 'pandas._libs.tslibs.timedeltas.Timedelta'>    15549
Name: count, dtype: int64

In [2014]:
from datetime import timedelta
import pandas as pd

# -------------------------------
# 1. Clean Date
# -------------------------------
Dialers['Date'] = pd.to_datetime(
    Dialers['Date'],
    format='%Y-%m-%d',
    errors='coerce'
)
Dialers


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-04-03,597 suresh,918918257549,not connected,19:44:43,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
1,Tata,2026-04-03,597 suresh,919727948999,not connected,19:44:16,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
2,Tata,2026-04-03,597 suresh,918367656635,not connected,19:43:45,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
3,Tata,2026-04-03,597 suresh,918303691062,not connected,19:43:21,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
4,Tata,2026-04-03,597 suresh,919864480486,not connected,19:42:51,0 days 00:00:09,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16503,Stringee,2026-04-03,esha,919177092480,not connected,10:49:07,0 days 00:00:40,0 days,0 days 00:00:35,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16504,Stringee,2026-04-03,esha,918010543210,not connected,10:48:28,0 days 00:00:20,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16505,Stringee,2026-04-03,esha,919248792487,not connected,10:47:09,0 days 00:00:59,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16506,Stringee,2026-04-03,esha,917032320554,not connected,10:46:38,0 days 00:00:24,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX


In [2015]:
# -------------------------------
# 2. Clean Call Start Time
# -------------------------------
# Ensure it's proper time first
Dialers['Call Start Time'] = pd.to_datetime(
    Dialers['Call Start Time'],
    errors='coerce'
).dt.strftime('%H:%M:%S')

# Combine Date + Time → full datetime
Dialers['Call Start Time'] = pd.to_datetime(
    Dialers['Date'].dt.strftime('%Y-%m-%d') + ' ' + Dialers['Call Start Time'],
    errors='coerce'
)
Dialers

C:\Users\Akhil\AppData\Local\Temp\ipykernel_17724\2121808340.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Dialers['Call Start Time'] = pd.to_datetime(


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-04-03,597 suresh,918918257549,not connected,2026-04-03 19:44:43,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
1,Tata,2026-04-03,597 suresh,919727948999,not connected,2026-04-03 19:44:16,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
2,Tata,2026-04-03,597 suresh,918367656635,not connected,2026-04-03 19:43:45,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
3,Tata,2026-04-03,597 suresh,918303691062,not connected,2026-04-03 19:43:21,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
4,Tata,2026-04-03,597 suresh,919864480486,not connected,2026-04-03 19:42:51,0 days 00:00:09,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16503,Stringee,2026-04-03,esha,919177092480,not connected,2026-04-03 10:49:07,0 days 00:00:40,0 days,0 days 00:00:35,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16504,Stringee,2026-04-03,esha,918010543210,not connected,2026-04-03 10:48:28,0 days 00:00:20,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16505,Stringee,2026-04-03,esha,919248792487,not connected,2026-04-03 10:47:09,0 days 00:00:59,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16506,Stringee,2026-04-03,esha,917032320554,not connected,2026-04-03 10:46:38,0 days 00:00:24,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX


In [2016]:
# -------------------------------
# 3. Clean Duration (IMPORTANT FIX)
# -------------------------------
Dialers['Total Call Duration'] = pd.to_timedelta(
    Dialers['Total Call Duration'],
    errors='coerce'
).fillna(pd.Timedelta(0))
Dialers

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-04-03,597 suresh,918918257549,not connected,2026-04-03 19:44:43,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
1,Tata,2026-04-03,597 suresh,919727948999,not connected,2026-04-03 19:44:16,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
2,Tata,2026-04-03,597 suresh,918367656635,not connected,2026-04-03 19:43:45,0 days 00:00:07,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
3,Tata,2026-04-03,597 suresh,918303691062,not connected,2026-04-03 19:43:21,0 days 00:00:05,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
4,Tata,2026-04-03,597 suresh,919864480486,not connected,2026-04-03 19:42:51,0 days 00:00:09,0 days,0 days 00:00:00,Tata,suresh@nexora.live,E-1900,Suresh,OJT,Saif,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16503,Stringee,2026-04-03,esha,919177092480,not connected,2026-04-03 10:49:07,0 days 00:00:40,0 days,0 days 00:00:35,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16504,Stringee,2026-04-03,esha,918010543210,not connected,2026-04-03 10:48:28,0 days 00:00:20,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16505,Stringee,2026-04-03,esha,919248792487,not connected,2026-04-03 10:47:09,0 days 00:00:59,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX
16506,Stringee,2026-04-03,esha,917032320554,not connected,2026-04-03 10:46:38,0 days 00:00:24,0 days,0 days 00:00:00,Stringee,esha@nexora.live,E-1044,Isha Mehta,Pull Back,Swathi,TradeX


In [2017]:
# -------------------------------
# 4. Sort data
# -------------------------------
Dialers = Dialers.sort_values(
    by=['Date', 'CRM ID', 'Call Start Time']
).reset_index(drop=True)
Dialers

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-04-03,599 aamir,919148696641,not connected,2026-04-03 10:05:19,0 days 00:00:03,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX
1,Tata,2026-04-03,599 aamir,919523929235,not connected,2026-04-03 10:05:35,0 days 00:00:08,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX
2,Tata,2026-04-03,599 aamir,919636559741,not connected,2026-04-03 10:05:49,0 days 00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX
3,Tata,2026-04-03,599 aamir,918015443933,not connected,2026-04-03 10:06:24,0 days 00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX
4,Tata,2026-04-03,599 aamir,919510738298,not connected,2026-04-03 10:07:00,0 days 00:00:07,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15544,Tata,2026-04-03,zaid,919897667724,not connected,2026-04-03 18:14:41,0 days 00:00:27,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX
15545,Tata,2026-04-03,zaid,919822972243,not connected,2026-04-03 18:20:14,0 days 00:00:31,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX
15546,Tata,2026-04-03,zaid,919356146200,not connected,2026-04-03 18:21:30,0 days 00:00:14,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX
15547,Tata,2026-04-03,zaid,916388348128,connected,2026-04-03 18:22:42,0 days 00:00:27,0 days 00:00:04,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX


In [2018]:
# -------------------------------
# 5. Initialize columns
# -------------------------------
Dialers['Call Gap'] = 'No'
Dialers['Gap Duration'] = '00:00:00'

# -------------------------------
# 6. Working hours
# -------------------------------
start_time = pd.to_datetime('09:30:00').time()
end_time = pd.to_datetime('18:30:00').time()
Dialers

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-04-03,599 aamir,919148696641,not connected,2026-04-03 10:05:19,0 days 00:00:03,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-04-03,599 aamir,919523929235,not connected,2026-04-03 10:05:35,0 days 00:00:08,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
2,Tata,2026-04-03,599 aamir,919636559741,not connected,2026-04-03 10:05:49,0 days 00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
3,Tata,2026-04-03,599 aamir,918015443933,not connected,2026-04-03 10:06:24,0 days 00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
4,Tata,2026-04-03,599 aamir,919510738298,not connected,2026-04-03 10:07:00,0 days 00:00:07,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15544,Tata,2026-04-03,zaid,919897667724,not connected,2026-04-03 18:14:41,0 days 00:00:27,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:00
15545,Tata,2026-04-03,zaid,919822972243,not connected,2026-04-03 18:20:14,0 days 00:00:31,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:00
15546,Tata,2026-04-03,zaid,919356146200,not connected,2026-04-03 18:21:30,0 days 00:00:14,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:00
15547,Tata,2026-04-03,zaid,916388348128,connected,2026-04-03 18:22:42,0 days 00:00:27,0 days 00:00:04,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:00


In [2019]:
# -------------------------------
# 7. Gap Calculation Loop
# -------------------------------
for i in range(1, len(Dialers)):

    same_crm = Dialers.loc[i, 'CRM ID'] == Dialers.loc[i - 1, 'CRM ID']
    same_date = Dialers.loc[i, 'Date'] == Dialers.loc[i - 1, 'Date']

    if same_crm and same_date:

        current_time = Dialers.loc[i, 'Call Start Time'].time()
        previous_time = Dialers.loc[i - 1, 'Call Start Time'].time()

        previous_end = (
            Dialers.loc[i - 1, 'Call Start Time'] +
            Dialers.loc[i - 1, 'Total Call Duration']
        )

        gap_duration = Dialers.loc[i, 'Call Start Time'] - previous_end

        # Fix negative gaps
        if gap_duration.total_seconds() < 0:
            gap_duration = timedelta(0)

        # Format gap duration
        total_seconds = int(gap_duration.total_seconds())
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60

        Dialers.loc[i, 'Gap Duration'] = f"{hours:02}:{minutes:02}:{seconds:02}"

        # Apply business rule
        if start_time <= current_time <= end_time and start_time <= previous_time <= end_time:
            Dialers.loc[i, 'Call Gap'] = 'Yes' if gap_duration > timedelta(minutes=1) else 'No'
Dialers

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-04-03,599 aamir,919148696641,not connected,2026-04-03 10:05:19,0 days 00:00:03,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-04-03,599 aamir,919523929235,not connected,2026-04-03 10:05:35,0 days 00:00:08,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:13
2,Tata,2026-04-03,599 aamir,919636559741,not connected,2026-04-03 10:05:49,0 days 00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:06
3,Tata,2026-04-03,599 aamir,918015443933,not connected,2026-04-03 10:06:24,0 days 00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:04
4,Tata,2026-04-03,599 aamir,919510738298,not connected,2026-04-03 10:07:00,0 days 00:00:07,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15544,Tata,2026-04-03,zaid,919897667724,not connected,2026-04-03 18:14:41,0 days 00:00:27,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:35
15545,Tata,2026-04-03,zaid,919822972243,not connected,2026-04-03 18:20:14,0 days 00:00:31,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:05:06
15546,Tata,2026-04-03,zaid,919356146200,not connected,2026-04-03 18:21:30,0 days 00:00:14,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:45
15547,Tata,2026-04-03,zaid,916388348128,connected,2026-04-03 18:22:42,0 days 00:00:27,0 days 00:00:04,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:58


In [2020]:
# -------------------------------
# 8. (Optional) Final formatting
# -------------------------------
Dialers['Total Call Duration'] = Dialers['Total Call Duration'].astype(str).str[-8:]

# -------------------------------
# DONE
# -------------------------------
Dialers

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-04-03,599 aamir,919148696641,not connected,2026-04-03 10:05:19,00:00:03,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-04-03,599 aamir,919523929235,not connected,2026-04-03 10:05:35,00:00:08,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:13
2,Tata,2026-04-03,599 aamir,919636559741,not connected,2026-04-03 10:05:49,00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:06
3,Tata,2026-04-03,599 aamir,918015443933,not connected,2026-04-03 10:06:24,00:00:31,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:04
4,Tata,2026-04-03,599 aamir,919510738298,not connected,2026-04-03 10:07:00,00:00:07,0 days 00:00:00,0 days,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15544,Tata,2026-04-03,zaid,919897667724,not connected,2026-04-03 18:14:41,00:00:27,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:35
15545,Tata,2026-04-03,zaid,919822972243,not connected,2026-04-03 18:20:14,00:00:31,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:05:06
15546,Tata,2026-04-03,zaid,919356146200,not connected,2026-04-03 18:21:30,00:00:14,0 days 00:00:00,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:45
15547,Tata,2026-04-03,zaid,916388348128,connected,2026-04-03 18:22:42,00:00:27,0 days 00:00:04,0 days,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:58


In [2021]:
Dialers.to_csv(r"C:\Users\Akhil\Downloads\project\TradeX_raw\Dialers_final.csv", index=False)

In [2022]:
Dialers.isnull().sum()

C:\Users\Akhil\AppData\Roaming\Python\Python313\site-packages\IPython\core\displayhook.py:292: UserWarning: Output cache limit (currently 1000 entries) hit.
Flushing oldest 200 entries.
  warn('Output cache limit (currently {sz} entries) hit.\n'


Source                 0
Date                   0
Dialer Name            0
Number                 7
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
Dialer                 0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
Call Gap               0
Gap Duration           0
dtype: int64

In [2023]:
import re
invalid_values = []

# Function to convert mixed formats to seconds
def to_seconds(value):
    try:
        if isinstance(value, pd.Timedelta):
            return int(value.total_seconds())  # Convert timedelta to seconds
        elif re.match(r"^\d{1,2}:\d{1,2}:\d{1,2}$", str(value)):
            parts = list(map(int, value.split(':')))
            while len(parts) < 3:
                parts.insert(0, 0)  
            return parts[0] * 3600 + parts[1] * 60 + parts[2]
        elif str(value).isdigit():
            return int(value)
        else:
            invalid_values.append(value)
            return value
    except Exception:
        invalid_values.append(value)
        return value

# Replace missing values with empty strings
Dialers['Talk Time'] = Dialers['Talk Time'].fillna('')
Dialers['Hold Time'] = Dialers['Hold Time'].fillna('')
Dialers['Total Call Duration'] = Dialers['Total Call Duration'].fillna('')
Dialers['Gap Duration'] = Dialers['Gap Duration'].fillna('')

# Convert Talk Time, Hold Time, and Total Call Duration to seconds
Dialers['Talk Time (seconds)'] = Dialers['Talk Time'].apply(to_seconds)
Dialers['Hold Time (seconds)'] = Dialers['Hold Time'].apply(to_seconds)
Dialers['Total Duration (seconds)'] = Dialers['Total Call Duration'].apply(to_seconds)
Dialers['Gap Duration (seconds)'] = Dialers['Gap Duration'].apply(to_seconds)

# Notify invalid values
if invalid_values:
    print("Invalid values found in 'Talk Time', 'Total Call Duration', or 'Hold Time':, or 'Call Gap Duration':")
    print(invalid_values)

In [2058]:
Dialers[Dialers['Source'] == 'Stringee']['Talk Time (seconds)'].unique()

array([   0,   55,   34,  267,  244,   26,    9,   20,  140,   52,   50,
         21,   72,   14,   18,  121, 1566,   15,   92,    8,    7,   96,
        103,  119,   19,   16,  120,   28,  125,   13,  148,   77,   22,
         43,   63,   44,   40,   23,   33,  107,    5,   32,  214,    6,
        489,   38,  452,   46,   62,   10,    3,   39,  246,   29,  841,
        953,  369,  167,   41,   64,   17,   24,   11,   31,   25,  344,
         85,    2,   35,  225,  117,   78,   89,  268,   48,   59,   69,
         12,  128, 2447,  340,  258,   60,   58,  263,   56,    1,  102,
          4,   30,   51,   36,  136,  201,  141,   71,   91,   42,   98,
         57,   87,   65,  182,  134,   86,  108,   53,   37,  153,  176,
        207,   94,  210,   45,   27,  243,   83,   76,   68,   47,  261,
         54,  147,   66,  101,   84,  154,   95,  113,   67,  110,  916,
        122,   88,   90,  283,  539,  834,  324,  133,  145,  334,   80,
         74,  209,  277,  146,   61,   97,  259,  2

In [2059]:
Dialers.dtypes

Source                               object
Date                         datetime64[ns]
Dialer Name                          object
Number                               object
Call Status                          object
Call Start Time              datetime64[ns]
Total Call Duration                  object
Talk Time                   timedelta64[ns]
Hold Time                   timedelta64[ns]
Dialer                               object
CRM ID                               object
Employee code                        object
Full Name                            object
Pool                                 object
TL                                   object
Vertical                             object
Call Gap                             object
Gap Duration                         object
Talk Time (seconds)                   int64
Hold Time (seconds)                   int64
Total Duration (seconds)              int64
Gap Duration (seconds)                int64
dtype: object

In [2060]:
Dialers.isnull().sum()

Source                      0
Date                        0
Dialer Name                 0
Number                      0
Call Status                 0
Call Start Time             0
Total Call Duration         0
Talk Time                   0
Hold Time                   0
Dialer                      0
CRM ID                      0
Employee code               0
Full Name                   0
Pool                        0
TL                          0
Vertical                    0
Call Gap                    0
Gap Duration                0
Talk Time (seconds)         0
Hold Time (seconds)         0
Total Duration (seconds)    0
Gap Duration (seconds)      0
dtype: int64

### Change


In [2061]:
# df['Total_Duration'] = pd.to_timedelta(df['Total_Duration'], errors='coerce')

In [2062]:
# Group by 'CRM ID' and 'Date' and calculate 
A = Dialers.groupby(['CRM ID', 'Date']).agg(
    Total_Dialed_Calls=('Call Status', 'count'),
    Unique_Dialed_Numbers=('Number', 'nunique'),
    Total_Connected_Calls=('Call Status', lambda x: (x == 'connected').sum()),
    Total_Number_of_Call_Gap=('Call Gap', lambda x: (x == 'Yes').sum()),
    Total_Call_GT_30=('Talk Time (seconds)', lambda x: ((Dialers.loc[x.index, 'Call Status'] == 'connected') & (x > 30)).sum()),
    Total_Duration=('Total Duration (seconds)', 'sum'),
    Total_Talk_Time=('Talk Time (seconds)', lambda x: x[Dialers.loc[x.index, 'Call Status'] == 'connected'].sum()),
    Total_Talk_Time_GT_30=('Talk Time (seconds)', lambda x: x[(Dialers.loc[x.index, 'Call Status'] == 'connected') & (x > 30)].sum()),
    Total_Connected_Hold_Time=('Hold Time (seconds)', lambda x: x[Dialers.loc[x.index, 'Call Status'] == 'connected'].sum()), 
    Total_Gap_Duration=('Gap Duration (seconds)', 'sum')
).reset_index()

# Fix: Subtract 1hr only if greater, else keep the original value
A['Total_Gap_Duration'] = A['Total_Gap_Duration'].apply(lambda x: x - 3600 if x > 3600 else x)


A['Avg_Gap_per_call'] = A['Total_Gap_Duration'] / A['Total_Dialed_Calls']

# New: Gap Duration After Leverage (give 45 seconds per call, subtract from actual gap used, add to 0 if negative)
A['Gap Duration After Leverage'] = (A['Total_Gap_Duration'] - (A['Total_Dialed_Calls'] * 45)).clip(lower=0)

# Recalculate Login Hours after updated gap
A['Login Hours'] = A['Total_Duration'] + A['Total_Gap_Duration'] + 3600
A['Login Hours'] = pd.to_timedelta(A['Login Hours'], unit='s')
A['Login Hours'] = A['Login Hours'].apply(lambda x: str(x).split()[-1])

# Convert 'Login Hours' string (hh:mm:ss) to timedelta
A['Login Hours (Timedelta)'] = pd.to_timedelta(A['Login Hours'])

# Defining attendance logic
def mark_attendance(td):
    if td < pd.Timedelta(hours=4, minutes=30):
        return 'Absent'
    elif td < pd.Timedelta(hours=6):
        return 'Half Day'
    elif td < pd.Timedelta(hours=8, minutes=30):
        return 'Warning'
    else:
        return 'Present'


A['Attendance'] = A['Login Hours (Timedelta)'].apply(mark_attendance)


A.drop(columns='Login Hours (Timedelta)', inplace=True)


A


,CRM ID,Date,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,Total_Number_of_Call_Gap,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,aamir@nexora.live,2026-04-03,303,281,51,35,17,6955,2107,1704,0,17781,58.683168,4146,07:52:16,Warning
1,aaryan@nexora.live,2026-04-03,254,237,52,43,23,11972,6535,6179,0,20432,80.440945,9002,10:00:04,Present
2,abhinav.a@nexora.live,2026-04-03,170,137,34,95,6,5407,777,428,0,24744,145.552941,17094,09:22:31,Present
3,abhishek.k@nexora.live,2026-04-03,255,150,74,40,39,17308,12303,11849,0,15104,59.231373,3629,10:00:12,Present
4,adharv@nexora.live,2026-04-03,292,225,94,78,35,11714,5756,4984,0,21103,72.270548,7963,10:06:57,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,tarun@nexora.live,2026-04-03,151,148,47,49,23,6636,3472,3167,0,21489,142.311258,14694,08:48:45,Present
65,vaibhav@nexora.live,2026-04-03,254,210,52,69,26,9812,3930,3545,0,21557,84.870079,10127,09:42:49,Present
66,waman@nexora.live,2026-04-03,202,174,58,57,30,12146,7919,7557,0,18258,90.386139,9168,09:26:44,Present
67,yatin@nexora.live,2026-04-03,250,238,16,50,7,2588,905,823,0,18224,72.896000,6974,06:46:52,Warning


In [2028]:
AB = A['Attendance'].unique()
AB

array(['Warning', 'Present', 'Half Day'], dtype=object)

In [2029]:
ref

,Dialer Name,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,223abhishek,Knowlarity,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
1,223 abhishek,Tata,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
3,412 anuj,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit,Tata,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
...,...,...,...,...,...,...,...,...
202,mohit,Tata,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,alina,Tata,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,siddhesh,Tata,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX
205,reeya,Tata,reeya@nexora.live,E-2126,Reeya,OJT 1,Rehan,TradeX


In [2030]:
# Filter unique CRM ID and select specific columns
unique_crm_ref = ref.drop_duplicates(subset=['CRM ID'])[['CRM ID','Employee code', 'Full Name','Pool', 'TL','Vertical']]

unique_crm_ref

,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
6,amrut.s@nexora.live,E-1282,Amrut Shivaji,Pull back-WFH,Swathi,TradeX
7,ashutosh@nexora.live,E-1253,Ashutosh Singh,1,Saif,TradeX
...,...,...,...,...,...,...
201,kanika@nexora.live,E-2132,Kanika,OJT 1,Rehan,TradeX
202,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX


In [2031]:
# Extract unique dates from DataFrame A
all_dates = A['Date'].unique()

# Create a DataFrame with all CRM IDs from unique_crm_ref and all dates
date_crm_combinations = pd.MultiIndex.from_product(
    [unique_crm_ref['CRM ID'], all_dates],
    names=['CRM ID', 'Date']
).to_frame(index=False)



merged = date_crm_combinations.merge(
    A,
    how='left',
    on=['CRM ID', 'Date']
).fillna(0)  


In [2032]:
merged

,CRM ID,Date,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,Total_Number_of_Call_Gap,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,abhishek.k@nexora.live,2026-04-03,255.0,150.0,74.0,40.0,39.0,17308.0,12303.0,11849.0,0.0,15104.0,59.231373,3629.0,10:00:12,Present
1,adharv@nexora.live,2026-04-03,292.0,225.0,94.0,78.0,35.0,11714.0,5756.0,4984.0,0.0,21103.0,72.270548,7963.0,10:06:57,Present
2,advit@nexora.live,2026-04-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
3,amrut.s@nexora.live,2026-04-03,206.0,203.0,181.0,176.0,49.0,8652.0,5701.0,3871.0,0.0,18065.0,87.694175,8795.0,08:25:17,Warning
4,ashutosh@nexora.live,2026-04-03,160.0,125.0,26.0,74.0,15.0,7576.0,4352.0,4215.0,0.0,22603.0,141.268750,15403.0,09:22:59,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,kanika@nexora.live,2026-04-03,295.0,281.0,45.0,55.0,9.0,8658.0,2610.0,2290.0,0.0,20296.0,68.800000,7021.0,09:02:34,Present
110,mohit@nexora.live,2026-04-03,225.0,186.0,54.0,41.0,26.0,9172.0,4765.0,4326.0,0.0,19614.0,87.173333,9489.0,08:59:46,Present
111,alina@nexora.live,2026-04-03,297.0,280.0,59.0,73.0,29.0,12542.0,5880.0,5388.0,0.0,24064.0,81.023569,10699.0,11:10:06,Present
112,siddhesh@nexora.live,2026-04-03,222.0,215.0,52.0,31.0,4.0,4276.0,760.0,307.0,0.0,17389.0,78.328829,7399.0,07:01:05,Warning


In [2033]:
# Outer-merge with unique_crm_ref to retain all CRM IDs
merged_df = unique_crm_ref.merge(
    merged,
    how='outer',
    on='CRM ID'
)

In [2034]:
merged_df

,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Date,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,Deepa.negi@nexora.live,E-1443,Suja Basnet,Meta,Radhika,TradeX,2026-04-03,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
1,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,2026-04-03,303.0,281.0,51.0,...,17.0,6955.0,2107.0,1704.0,0.0,17781.0,58.683168,4146.0,07:52:16,Warning
2,aaryan@nexora.live,E-2124,Aaryan,OJT 1,Rehan,TradeX,2026-04-03,254.0,237.0,52.0,...,23.0,11972.0,6535.0,6179.0,0.0,20432.0,80.440945,9002.0,10:00:04,Present
3,abhinav.a@nexora.live,E-1356,Abhinav,1,Saif,TradeX,2026-04-03,170.0,137.0,34.0,...,6.0,5407.0,777.0,428.0,0.0,24744.0,145.552941,17094.0,09:22:31,Present
4,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX,2026-04-03,255.0,150.0,74.0,...,39.0,17308.0,12303.0,11849.0,0.0,15104.0,59.231373,3629.0,10:00:12,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,vandana@nexora.live,E - 1677,Vandana,Dialer,Rehan,TradeX,2026-04-03,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
110,vicky.v@nexora.live,E-1167,Vishal,Customer care,Vikas,TradeX,2026-04-03,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
111,waman@nexora.live,E-1252,Waman Avhad,1,Saif,TradeX,2026-04-03,202.0,174.0,58.0,...,30.0,12146.0,7919.0,7557.0,0.0,18258.0,90.386139,9168.0,09:26:44,Present
112,yatin@nexora.live,E-1906,Yatin,OJT,Saif,TradeX,2026-04-03,250.0,238.0,16.0,...,7.0,2588.0,905.0,823.0,0.0,18224.0,72.896000,6974.0,06:46:52,Warning


In [2035]:
# Replace NaN or missing values with 0
columns_to_format = ['Total_Dialed_Calls', 'Unique_Dialed_Numbers','Total_Connected_Calls','Total_Number_of_Call_Gap', 'Total_Call_GT_30','Total_Duration','Total_Talk_Time','Total_Talk_Time_GT_30','Total_Connected_Hold_Time','Total_Gap_Duration']
merged_df[columns_to_format] = merged_df[columns_to_format].fillna(0)

# Convert specified columns to integers
merged_df[columns_to_format] = merged_df[columns_to_format].astype(int)

# Reorder columns
formatted_df = merged_df[['Date', 'Pool', 'TL', 'CRM ID','Employee code', 'Full Name', 'Vertical'] + [col for col in merged_df.columns if col not in ['Date', 'Pool', 'TL', 'CRM ID','Employee code', 'Full Name', 'Vertical']]]

formatted_df

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,2026-04-03,Meta,Radhika,Deepa.negi@nexora.live,E-1443,Suja Basnet,TradeX,0,0,0,...,0,0,0,0,0,0,0.000000,0.0,0,0
1,2026-04-03,OJT,Saif,aamir@nexora.live,E-1911,Aamir,TradeX,303,281,51,...,17,6955,2107,1704,0,17781,58.683168,4146.0,07:52:16,Warning
2,2026-04-03,OJT 1,Rehan,aaryan@nexora.live,E-2124,Aaryan,TradeX,254,237,52,...,23,11972,6535,6179,0,20432,80.440945,9002.0,10:00:04,Present
3,2026-04-03,1,Saif,abhinav.a@nexora.live,E-1356,Abhinav,TradeX,170,137,34,...,6,5407,777,428,0,24744,145.552941,17094.0,09:22:31,Present
4,2026-04-03,Meta,Radhika,abhishek.k@nexora.live,E-1266,Abhishek Kumar,TradeX,255,150,74,...,39,17308,12303,11849,0,15104,59.231373,3629.0,10:00:12,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026-04-03,Dialer,Rehan,vandana@nexora.live,E - 1677,Vandana,TradeX,0,0,0,...,0,0,0,0,0,0,0.000000,0.0,0,0
110,2026-04-03,Customer care,Vikas,vicky.v@nexora.live,E-1167,Vishal,TradeX,0,0,0,...,0,0,0,0,0,0,0.000000,0.0,0,0
111,2026-04-03,1,Saif,waman@nexora.live,E-1252,Waman Avhad,TradeX,202,174,58,...,30,12146,7919,7557,0,18258,90.386139,9168.0,09:26:44,Present
112,2026-04-03,OJT,Saif,yatin@nexora.live,E-1906,Yatin,TradeX,250,238,16,...,7,2588,905,823,0,18224,72.896000,6974.0,06:46:52,Warning


In [2036]:
formatted_df['Total_Connected_Calls'].unique()


array([  0,  51,  52,  34,  74,  94,  59,  61, 181,  50,  26,  39,  47,
        42,  40,  57,  27,  70,  68,  32,  45,  44,  62,  20,  15,  56,
        54,  58,  33,  13,  89,  67,  86,  49,  73,  41,  72,  22,  38,
        16])

In [2037]:
formatted_df.isnull().sum()

Date                           0
Pool                           1
TL                             1
CRM ID                         0
Employee code                  0
Full Name                      0
Vertical                       0
Total_Dialed_Calls             0
Unique_Dialed_Numbers          0
Total_Connected_Calls          0
Total_Number_of_Call_Gap       0
Total_Call_GT_30               0
Total_Duration                 0
Total_Talk_Time                0
Total_Talk_Time_GT_30          0
Total_Connected_Hold_Time      0
Total_Gap_Duration             0
Avg_Gap_per_call               0
Gap Duration After Leverage    0
Login Hours                    0
Attendance                     0
dtype: int64

In [2038]:
def seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return None  # or "00:00:00" if you prefer
    total_seconds = round(seconds)
    hours = total_seconds // 3600
    remaining = total_seconds % 3600
    minutes = remaining // 60
    seconds = remaining % 60
    return f"{hours:02}:{minutes:02}:{seconds:02}"  # hh:mm:ss


columns_to_convert = [
    'Total_Duration', 
    'Total_Talk_Time', 
    'Total_Talk_Time_GT_30', 
    'Total_Connected_Hold_Time', 
    'Total_Gap_Duration',
    'Avg_Gap_per_call',
    'Gap Duration After Leverage'
]

for col in columns_to_convert:
    try:
        formatted_df[col] = formatted_df[col].apply(seconds_to_hhmmss)
    except Exception as e:
        print(f"Error in column: {col}")
        raise e  # re-raise the error so you still get the traceback


In [2039]:
# from datetime import timedelta

# def seconds_to_hhmmss(seconds):
#     total_seconds = round(seconds)
#     hours = total_seconds // 3600
#     remaining = total_seconds % 3600
#     minutes = remaining // 60
#     seconds = remaining % 60
#     return f"{hours:02}:{minutes:02}:{seconds:02}"  # hh:mm:ss


# columns_to_convert = [
#     'Total_Duration', 
#     'Total_Talk_Time', 
#     'Total_Talk_Time_GT_30', 
#     'Total_Connected_Hold_Time', 
#     'Total_Gap_Duration',
#     'Avg_Gap_per_call',
#     'Gap Duration After Leverage'
# ]

# for col in columns_to_convert:
#     formatted_df[col] = formatted_df[col].apply(seconds_to_hhmmss)	


In [2040]:
formatted_df = formatted_df.sort_values(by=['Date','CRM ID'])

In [2041]:
formatted_df

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,2026-04-03,Meta,Radhika,Deepa.negi@nexora.live,E-1443,Suja Basnet,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,0,0
1,2026-04-03,OJT,Saif,aamir@nexora.live,E-1911,Aamir,TradeX,303,281,51,...,17,01:55:55,00:35:07,00:28:24,00:00:00,04:56:21,00:00:59,01:09:06,07:52:16,Warning
2,2026-04-03,OJT 1,Rehan,aaryan@nexora.live,E-2124,Aaryan,TradeX,254,237,52,...,23,03:19:32,01:48:55,01:42:59,00:00:00,05:40:32,00:01:20,02:30:02,10:00:04,Present
3,2026-04-03,1,Saif,abhinav.a@nexora.live,E-1356,Abhinav,TradeX,170,137,34,...,6,01:30:07,00:12:57,00:07:08,00:00:00,06:52:24,00:02:26,04:44:54,09:22:31,Present
4,2026-04-03,Meta,Radhika,abhishek.k@nexora.live,E-1266,Abhishek Kumar,TradeX,255,150,74,...,39,04:48:28,03:25:03,03:17:29,00:00:00,04:11:44,00:00:59,01:00:29,10:00:12,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026-04-03,Dialer,Rehan,vandana@nexora.live,E - 1677,Vandana,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,0,0
110,2026-04-03,Customer care,Vikas,vicky.v@nexora.live,E-1167,Vishal,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,0,0
111,2026-04-03,1,Saif,waman@nexora.live,E-1252,Waman Avhad,TradeX,202,174,58,...,30,03:22:26,02:11:59,02:05:57,00:00:00,05:04:18,00:01:30,02:32:48,09:26:44,Present
112,2026-04-03,OJT,Saif,yatin@nexora.live,E-1906,Yatin,TradeX,250,238,16,...,7,00:43:08,00:15:05,00:13:43,00:00:00,05:03:44,00:01:13,01:56:14,06:46:52,Warning


In [2042]:
formatted_df['Attendance'].unique()

array([0, 'Warning', 'Present', 'Half Day'], dtype=object)

In [2043]:

formatted_df['Login Hours'] = formatted_df['Login Hours'].apply(lambda x: '00:00:00' if x == 0 else x)

formatted_df['Attendance'] = formatted_df['Attendance'].apply(lambda x: 'Absent' if x == 0 else x)


In [2044]:
formatted_df

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,2026-04-03,Meta,Radhika,Deepa.negi@nexora.live,E-1443,Suja Basnet,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,Absent
1,2026-04-03,OJT,Saif,aamir@nexora.live,E-1911,Aamir,TradeX,303,281,51,...,17,01:55:55,00:35:07,00:28:24,00:00:00,04:56:21,00:00:59,01:09:06,07:52:16,Warning
2,2026-04-03,OJT 1,Rehan,aaryan@nexora.live,E-2124,Aaryan,TradeX,254,237,52,...,23,03:19:32,01:48:55,01:42:59,00:00:00,05:40:32,00:01:20,02:30:02,10:00:04,Present
3,2026-04-03,1,Saif,abhinav.a@nexora.live,E-1356,Abhinav,TradeX,170,137,34,...,6,01:30:07,00:12:57,00:07:08,00:00:00,06:52:24,00:02:26,04:44:54,09:22:31,Present
4,2026-04-03,Meta,Radhika,abhishek.k@nexora.live,E-1266,Abhishek Kumar,TradeX,255,150,74,...,39,04:48:28,03:25:03,03:17:29,00:00:00,04:11:44,00:00:59,01:00:29,10:00:12,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026-04-03,Dialer,Rehan,vandana@nexora.live,E - 1677,Vandana,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,Absent
110,2026-04-03,Customer care,Vikas,vicky.v@nexora.live,E-1167,Vishal,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,Absent
111,2026-04-03,1,Saif,waman@nexora.live,E-1252,Waman Avhad,TradeX,202,174,58,...,30,03:22:26,02:11:59,02:05:57,00:00:00,05:04:18,00:01:30,02:32:48,09:26:44,Present
112,2026-04-03,OJT,Saif,yatin@nexora.live,E-1906,Yatin,TradeX,250,238,16,...,7,00:43:08,00:15:05,00:13:43,00:00:00,05:03:44,00:01:13,01:56:14,06:46:52,Warning


In [2045]:
# Convert 'Total Call Duration' to hh:mm:ss format
Dialers["Total Call Duration"] = Dialers["Total Call Duration"].apply(lambda x: str(x).split(" ")[-1])
Dialers['Number'] = "'" + Dialers['Number'].astype(str)


In [2046]:
D = Dialers.drop(columns=['Talk Time (seconds)','Hold Time (seconds)','Total Duration (seconds)','Gap Duration (seconds)','Dialer'])
D['Number'] = D['Number'].astype(str).str.split('.').str[0]
D

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-04-03,599 aamir,'919148696641,not connected,2026-04-03 10:05:19,00:00:03,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-04-03,599 aamir,'919523929235,not connected,2026-04-03 10:05:35,00:00:08,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:13
2,Tata,2026-04-03,599 aamir,'919636559741,not connected,2026-04-03 10:05:49,00:00:31,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:06
3,Tata,2026-04-03,599 aamir,'918015443933,not connected,2026-04-03 10:06:24,00:00:31,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:04
4,Tata,2026-04-03,599 aamir,'919510738298,not connected,2026-04-03 10:07:00,00:00:07,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15544,Tata,2026-04-03,zaid,'919897667724,not connected,2026-04-03 18:14:41,00:00:27,0 days 00:00:00,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:35
15545,Tata,2026-04-03,zaid,'919822972243,not connected,2026-04-03 18:20:14,00:00:31,0 days 00:00:00,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:05:06
15546,Tata,2026-04-03,zaid,'919356146200,not connected,2026-04-03 18:21:30,00:00:14,0 days 00:00:00,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:45
15547,Tata,2026-04-03,zaid,'916388348128,connected,2026-04-03 18:22:42,00:00:27,0 days 00:00:04,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:58


In [2047]:
D.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 0
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
Call Gap               0
Gap Duration           0
dtype: int64

In [2048]:
df = D.copy()

In [2049]:
unique_calls = df.drop_duplicates(subset=["Date", "CRM ID", "Number","Call Status"])[["Date", "CRM ID", "Number","Call Status","Pool","TL","Full Name"]]

In [2050]:
unique_calls

,Date,CRM ID,Number,Call Status,Pool,TL,Full Name
0,2026-04-03,aamir@nexora.live,'919148696641,not connected,OJT,Saif,Aamir
1,2026-04-03,aamir@nexora.live,'919523929235,not connected,OJT,Saif,Aamir
2,2026-04-03,aamir@nexora.live,'919636559741,not connected,OJT,Saif,Aamir
3,2026-04-03,aamir@nexora.live,'918015443933,not connected,OJT,Saif,Aamir
4,2026-04-03,aamir@nexora.live,'919510738298,not connected,OJT,Saif,Aamir
...,...,...,...,...,...,...,...
15544,2026-04-03,zaid.k@nexora.live,'919897667724,not connected,Pull Back,Parth,Zaid
15545,2026-04-03,zaid.k@nexora.live,'919822972243,not connected,Pull Back,Parth,Zaid
15546,2026-04-03,zaid.k@nexora.live,'919356146200,not connected,Pull Back,Parth,Zaid
15547,2026-04-03,zaid.k@nexora.live,'916388348128,connected,Pull Back,Parth,Zaid


In [2051]:
D.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 0
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
Call Gap               0
Gap Duration           0
dtype: int64

## Export


In [2052]:
save_path = r'C:\Users\Akhil\Downloads\project\TradeX_report'
formatted_df.to_csv(f'{save_path}\\Summary_03_04.csv', index=False)
D.to_csv(f'{save_path}\\Dialer_03_04.csv', index=False)
crm_id_null_df.to_csv(f'{save_path}\\Not_Found_Users_03_04.csv', index=False)

In [2053]:
df

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-04-03,599 aamir,'919148696641,not connected,2026-04-03 10:05:19,00:00:03,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-04-03,599 aamir,'919523929235,not connected,2026-04-03 10:05:35,00:00:08,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:13
2,Tata,2026-04-03,599 aamir,'919636559741,not connected,2026-04-03 10:05:49,00:00:31,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:06
3,Tata,2026-04-03,599 aamir,'918015443933,not connected,2026-04-03 10:06:24,00:00:31,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:04
4,Tata,2026-04-03,599 aamir,'919510738298,not connected,2026-04-03 10:07:00,00:00:07,0 days 00:00:00,0 days,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15544,Tata,2026-04-03,zaid,'919897667724,not connected,2026-04-03 18:14:41,00:00:27,0 days 00:00:00,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:35
15545,Tata,2026-04-03,zaid,'919822972243,not connected,2026-04-03 18:20:14,00:00:31,0 days 00:00:00,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:05:06
15546,Tata,2026-04-03,zaid,'919356146200,not connected,2026-04-03 18:21:30,00:00:14,0 days 00:00:00,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:45
15547,Tata,2026-04-03,zaid,'916388348128,connected,2026-04-03 18:22:42,00:00:27,0 days 00:00:04,0 days,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:58
